In [2]:
import pandas as pd
import numpy as np

file_path = '../data/raw/Sample - Superstore.csv'
df = pd.read_csv(file_path, encoding='cp1252')

print('Rows and columns:', df.shape)
df.head()

Rows and columns: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
print('column names:')
print(df.columns.tolist())

print('\ndata types:')
print(df.dtypes)

column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

data types:
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object


In [4]:
print('missing values per column:')
print(df.isnull().sum())
print('\ntotal missing values:', df.isnull().sum().sum())

missing values per column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

total missing values: 0


In [5]:
dup= df.duplicated().sum()
print('exact duplicate rows:',dup)

exact duplicate rows: 0


In [6]:
text_columns = df.select_dtypes(include='object').columns
for col in text_columns:
    print(f'{col}: {df[col].nunique()} unique values')

Order ID: 5009 unique values
Order Date: 1237 unique values
Ship Date: 1334 unique values
Ship Mode: 4 unique values
Customer ID: 793 unique values
Customer Name: 793 unique values
Segment: 3 unique values
Country: 1 unique values
City: 531 unique values
State: 49 unique values
Region: 4 unique values
Product ID: 1862 unique values
Category: 3 unique values
Sub-Category: 17 unique values
Product Name: 1850 unique values


In [7]:
columns_to_check = [
    'Ship Mode',
    'Segment',
    'Country',
    'Region',
    'Category',
    'Sub-Category'
]
for col in columns_to_check:
    print(f'\n{col}:')
    print(sorted(df[col].dropna().unique()))


Ship Mode:
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment:
['Consumer', 'Corporate', 'Home Office']

Country:
['United States']

Region:
['Central', 'East', 'South', 'West']

Category:
['Furniture', 'Office Supplies', 'Technology']

Sub-Category:
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']


In [8]:
numeric_columns = df.select_dtypes(include='number').columns
for col in numeric_columns:
    print(f'{col}: min = {df[col].min()}, max = {df[col].max()}')

Row ID: min = 1, max = 9994
Postal Code: min = 1040, max = 99301
Sales: min = 0.444, max = 22638.48
Quantity: min = 1, max = 14
Discount: min = 0.0, max = 0.8
Profit: min = -6599.978, max = 8399.976


In [9]:
order_dates = pd.to_datetime(
    df['Order Date'],
    format='%m/%d/%Y',
    errors='coerce'
)
ship_dates = pd.to_datetime(
    df['Ship Date'],
    format='%m/%d/%Y',
    errors='coerce'
)
print('invalid order dates:', order_dates.isna().sum())
print('invalid ship dates:', ship_dates.isna().sum())
print('ship date before order date:',
      (ship_dates < order_dates).sum())

invalid order dates: 0
invalid ship dates: 0
ship date before order date: 0


In [10]:
# leading or trailing spaces in text columns
text_columns = df.select_dtypes(include='object').columns
for col in text_columns:
    spaces = df[col].dropna().apply(
        lambda value: value != value.strip()
    ).sum()

    if spaces > 0:
        print(f'{col}: {spaces} values with extra spaces')
print('text whitespace check completed.')

Product Name: 16 values with extra spaces
text whitespace check completed.


In [11]:
mask = df['Product Name'].notna() & (
    df['Product Name'] != df['Product Name'].str.strip()
)
print('affected rows:', mask.sum())
display(df.loc[mask, ['Row ID', 'Product ID', 'Product Name']])

affected rows: 16


,Row ID,Product ID,Product Name
808,809,TEC-AC-10004901,Kensington SlimBlade Notebook Wireless Mouse w...
889,890,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...
1019,1020,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart
2169,2170,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...
2641,2642,TEC-AC-10000927,Anker Ultrathin Bluetooth Wireless Keyboard Al...
2759,2760,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart
2839,2840,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart
3590,3591,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart
3893,3894,TEC-AC-10004901,Kensington SlimBlade Notebook Wireless Mouse w...
4900,4901,OFF-ST-10001526,Iceberg Mobile Mega Data/Printer Cart


In [12]:
fault_log = [
    {
        'issue': 'Leading/trailing whitespace in Product Name',
        'evidence': '16 affected rows',
        'decision': 'Strip leading/trailing whitespace',
        'reason': 'Prevent inconsistent product-name grouping',
        'rows_affected': 16
    }
]

pd.DataFrame(fault_log)

,issue,evidence,decision,reason,rows_affected
0,Leading/trailing whitespace in Product Name,16 affected rows,Strip leading/trailing whitespace,Prevent inconsistent product-name grouping,16


In [13]:
df['Product Name'] = df['Product Name'].str.strip()

print(
    'rows with remaining extra spaces:',
    (
        df['Product Name'].notna()
        & (df['Product Name'] != df['Product Name'].str.strip())
    ).sum()
)

rows with remaining extra spaces: 0


In [14]:
print('postal code data type:', df['Postal Code'].dtype)
print('missing postal codes:', df['Postal Code'].isna().sum())
print('unique postal codes:', df['Postal Code'].nunique())

print('\nmost frequent postal codes:')
print(df['Postal Code'].value_counts().head(10))

postal code data type: int64
missing postal codes: 0
unique postal codes: 631

most frequent postal codes:
Postal Code
10035    263
10024    230
10009    229
94122    203
10011    193
94110    166
98105    165
19134    160
90049    151
98103    151
Name: count, dtype: int64


In [15]:
duplicate_order_products = df.duplicated(
    subset=['Order ID', 'Product ID'],
    keep=False
)
print(
    'rows involved in repeated Order ID + Product ID combinations:',
    duplicate_order_products.sum()
)

display(
    df.loc[
        duplicate_order_products,
        ['Order ID', 'Product ID', 'Product Name', 'Sales', 'Quantity']
    ].head(10)
)

rows involved in repeated Order ID + Product ID combinations: 16


,Order ID,Product ID,Product Name,Sales,Quantity
350,CA-2016-129714,OFF-PA-10001970,Xerox 1881,24.560,2
352,CA-2016-129714,OFF-PA-10001970,Xerox 1881,49.120,4
430,US-2016-123750,TEC-AC-10004659,Imation Secure+ Hardware Encrypted USB 2.0 Fla...,408.744,7
431,US-2016-123750,TEC-AC-10004659,Imation Secure+ Hardware Encrypted USB 2.0 Fla...,291.960,5
1300,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",572.760,6
1301,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",286.380,3
3183,CA-2017-152912,OFF-ST-10003208,Adjustable Depth Letter/Legal Cart,1633.140,9
3184,CA-2017-152912,OFF-ST-10003208,Adjustable Depth Letter/Legal Cart,544.380,3
3405,US-2014-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372,2
3406,US-2014-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372,2


In [16]:
display(
    df.loc[
        duplicate_order_products
    ].sort_values(['Order ID', 'Product ID'])
)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
6498,6499,CA-2015-103135,7/24/2015,7/28/2015,Standard Class,SS-20515,Shirley Schmidt,Home Office,United States,Louisville,...,40214,South,OFF-BI-10000069,Office Supplies,Binders,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",135.090,9,0.0,62.1414
6500,6501,CA-2015-103135,7/24/2015,7/28/2015,Standard Class,SS-20515,Shirley Schmidt,Home Office,United States,Louisville,...,40214,South,OFF-BI-10000069,Office Supplies,Binders,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",90.060,6,0.0,41.4276
350,351,CA-2016-129714,9/1/2016,9/3/2016,First Class,AB-10060,Adam Bellavance,Home Office,United States,New York City,...,10009,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,24.560,2,0.0,11.5432
352,353,CA-2016-129714,9/1/2016,9/3/2016,First Class,AB-10060,Adam Bellavance,Home Office,United States,New York City,...,10009,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,49.120,4,0.0,23.0864
1300,1301,CA-2016-137043,12/23/2016,12/25/2016,Second Class,LC-17140,Logan Currie,Consumer,United States,Springfield,...,22153,South,FUR-FU-10003664,Furniture,Furnishings,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",572.760,6,0.0,166.1004
1301,1302,CA-2016-137043,12/23/2016,12/25/2016,Second Class,LC-17140,Logan Currie,Consumer,United States,Springfield,...,22153,South,FUR-FU-10003664,Furniture,Furnishings,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",286.380,3,0.0,83.0502
9168,9169,CA-2016-140571,3/15/2016,3/19/2016,Standard Class,SJ-20125,Sanjit Jacobs,Home Office,United States,Jackson,...,39212,South,OFF-PA-10001954,Office Supplies,Paper,Xerox 1964,319.760,14,0.0,147.0896
9169,9170,CA-2016-140571,3/15/2016,3/19/2016,Standard Class,SJ-20125,Sanjit Jacobs,Home Office,United States,Jackson,...,39212,South,OFF-PA-10001954,Office Supplies,Paper,Xerox 1964,45.680,2,0.0,21.0128
7881,7882,CA-2017-118017,12/3/2017,12/6/2017,Second Class,LC-16870,Lena Cacioppo,Consumer,United States,Thornton,...,80229,West,TEC-AC-10002006,Technology,Accessories,Memorex Micro Travel Drive 16 GB,76.752,6,0.2,10.5534
7882,7883,CA-2017-118017,12/3/2017,12/6/2017,Second Class,LC-16870,Lena Cacioppo,Consumer,United States,Thornton,...,80229,West,TEC-AC-10002006,Technology,Accessories,Memorex Micro Travel Drive 16 GB,102.336,8,0.2,14.0712


In [17]:
print('sales less than or equal to zero:', (df['Sales']<=0).sum())
print('quantity less than or equal to zero:', (df['Quantity']<=0).sum())
print('discount below zero:', (df['Discount']<0).sum())
print('discount above one:', (df['Discount']>1).sum())

print('profit is missing:', df['Profit'].isna().sum())
print('negative profit rows:', (df['Profit']<0).sum())

sales less than or equal to zero: 0
quantity less than or equal to zero: 0
discount below zero: 0
discount above one: 0
profit is missing: 0
negative profit rows: 1871


In [18]:
print('duplicate Row IDs:', df['Row ID'].duplicated().sum())
print('unique Row IDs:', df['Row ID'].nunique())
print('minimum Row ID:', df['Row ID'].min())
print('maximum Row ID:', df['Row ID'].max())
print('total rows:', len(df))

duplicate Row IDs: 0
unique Row IDs: 9994
minimum Row ID: 1
maximum Row ID: 9994
total rows: 9994


In [19]:
order_dates=pd.to_datetime(df['Order Date'],format='%m/%d/%Y')
ship_dates=pd.to_datetime(df['Ship Date'],format='%m/%d/%Y')
shipping_days=(ship_dates-order_dates).dt.days

print('minimum shipping days:',shipping_days.min())
print('maximum shipping days:',shipping_days.max())
print('orders shipped on the same day:',(shipping_days==0).sum())
print('orders taking more than 7 days:',(shipping_days>7).sum())

display(
    df.loc[shipping_days>7,['Row ID','Order ID','Order Date','Ship Date','Ship Mode']]
)

minimum shipping days: 0
maximum shipping days: 7
orders shipped on the same day: 519
orders taking more than 7 days: 0


,Row ID,Order ID,Order Date,Ship Date,Ship Mode


In [22]:
text_columns=df.select_dtypes(include='object').columns
for col in text_columns:
    values=df[col].dropna().astype(str)

    extra_spaces=(values!=values.str.strip()).sum()
    mixed_case=values.str.lower().nunique()!=values.nunique()

    if extra_spaces>0 or mixed_case:
        print(
            col,
            '| Extra spaces:',extra_spaces,
            '| Possible case inconsistencies:',mixed_case
        )

In [23]:
print('total rows:',len(df))
print('total columns:',len(df.columns))
print('missing values:',df.isna().sum().sum())
print('exact duplicate rows:',df.duplicated().sum())
print('duplicate Row IDs:',df['Row ID'].duplicated().sum())
print('negative profit rows:',(df['Profit']<0).sum())

total rows: 9994
total columns: 21
missing values: 0
exact duplicate rows: 0
duplicate Row IDs: 0
negative profit rows: 1871


In [24]:
raw_df=pd.read_csv(file_path,encoding='cp1252')
before_rows=len(raw_df)
before_sales=raw_df['Sales'].sum()

after_rows=len(df)
after_sales=df['Sales'].sum()

print('before cleaning-rows:',before_rows)
print('before cleaning-total sales:',round(before_sales,2))
print('after cleaning-rows:',after_rows)
print('after cleaning-total sales:',round(after_sales,2))

before cleaning-rows: 9994
before cleaning-total sales: 2297200.86
after cleaning-rows: 9994
after cleaning-total sales: 2297200.86


In [26]:
validation=pd.DataFrame({
    'Metric':['Row count','Total sales'],
    'Before cleaning':[before_rows,before_sales],
    'After cleaning':[after_rows,after_sales]
})

validation['Difference']=(
    validation['After cleaning']-validation['Before cleaning']
)

display(validation.round(2))

,Metric,Before cleaning,After cleaning,Difference
0,Row count,9994.00,9994.00,0.0
1,Total sales,2297200.86,2297200.86,0.0


In [27]:
cleaning_log=pd.DataFrame([
    {
        'Issue':'Leading/trailing whitespace in Product Name',
        'Action':'Removed leading/trailing whitespace',
        'Reason':'Prevent inconsistent product-name grouping',
        'Rows affected':16
    }
])
display(cleaning_log)

,Issue,Action,Reason,Rows affected
0,Leading/trailing whitespace in Product Name,Removed leading/trailing whitespace,Prevent inconsistent product-name grouping,16


In [28]:
from pathlib import Path

report_dir=Path('../reports')
report_dir.mkdir(parents=True,exist_ok=True)

cleaning_log.to_csv(report_dir/'cleaning_log.csv',index=False)
validation.to_csv(report_dir/'validation.csv',index=False)

print('saved cleaning_log.csv')
print('saved validation.csv')
print('report folder:',report_dir.resolve())

saved cleaning_log.csv
saved validation.csv
report folder: C:\Users\Admin\week1\reports


In [29]:
df['Order Date']=pd.to_datetime(df['Order Date'],format='%m/%d/%Y')
df['Ship Date']=pd.to_datetime(df['Ship Date'],format='%m/%d/%Y')

df['Postal Code']=df['Postal Code'].astype('string')

print(df[['Order Date','Ship Date','Postal Code']].dtypes)

Order Date     datetime64[ns]
Ship Date      datetime64[ns]
Postal Code    string[python]
dtype: object


In [30]:
total_sales=df['Sales'].sum()
total_profit=df['Profit'].sum()
total_orders=df['Order ID'].nunique()
total_customers=df['Customer ID'].nunique()

print('total sales: $',round(total_sales,2))
print('total profit: $',round(total_profit,2))
print('unique orders:',total_orders)
print('unique customers:',total_customers)

total sales: $ 2297200.86
total profit: $ 286397.02
unique orders: 5009
unique customers: 793


In [31]:
profit_margin=(total_profit/total_sales)*100
print('overall profit margin:',round(profit_margin,2),'%')

overall profit margin: 12.47 %


In [32]:
region_summary=df.groupby('Region').agg(
    total_sales=('Sales','sum'),
    total_profit=('Profit','sum'),
    order_count=('Order ID','nunique')
).reset_index()

region_summary['profit_margin']=(
    region_summary['total_profit']/region_summary['total_sales']*100
)

region_summary=region_summary.sort_values(
    'total_profit',ascending=False
)
display(region_summary.round(2))

,Region,total_sales,total_profit,order_count,profit_margin
3,West,725457.82,108418.45,1611,14.94
1,East,678781.24,91522.78,1401,13.48
2,South,391721.90,46749.43,822,11.93
0,Central,501239.89,39706.36,1175,7.92


In [33]:
category_summary=df.groupby('Category').agg(
    total_sales=('Sales','sum'),
    total_profit=('Profit','sum'),
    quantity_sold=('Quantity','sum')
).reset_index()

category_summary['profit_margin']=(
    category_summary['total_profit']/category_summary['total_sales']*100
)

category_summary=category_summary.sort_values(
    'total_profit',ascending=False
)
display(category_summary.round(2))

,Category,total_sales,total_profit,quantity_sold,profit_margin
2,Technology,836154.03,145454.95,6939,17.40
1,Office Supplies,719047.03,122490.80,22906,17.04
0,Furniture,741999.80,18451.27,8028,2.49


In [34]:
subcategory_summary=df.groupby(['Category','Sub-Category']).agg(
    total_sales=('Sales','sum'),
    total_profit=('Profit','sum'),
    quantity_sold=('Quantity','sum')
).reset_index()
subcategory_summary['profit_margin']=(
    subcategory_summary['total_profit']
    /subcategory_summary['total_sales']*100
)
display(
    subcategory_summary.sort_values(
        'total_profit',ascending=True
    ).round(2)
)

,Category,Sub-Category,total_sales,total_profit,quantity_sold,profit_margin
3,Furniture,Tables,206965.53,-17725.48,1241,-8.56
0,Furniture,Bookcases,114880.00,-3472.56,868,-3.02
12,Office Supplies,Supplies,46673.54,-1189.10,647,-2.55
8,Office Supplies,Fasteners,3024.28,949.52,914,31.40
15,Technology,Machines,189238.63,3384.76,440,1.79
9,Office Supplies,Labels,12486.31,5546.25,1400,44.42
5,Office Supplies,Art,27118.79,6527.79,3000,24.07
7,Office Supplies,Envelopes,16476.40,6964.18,906,42.27
2,Furniture,Furnishings,91705.16,13059.14,3563,14.24
4,Office Supplies,Appliances,107532.16,18138.01,1729,16.87


In [35]:
discount_summary=df.groupby('Discount').agg(
    order_lines=('Row ID','count'),
    total_sales=('Sales','sum'),
    total_profit=('Profit','sum'),
    average_profit=('Profit','mean')
).reset_index()

display(discount_summary.round(2))

,Discount,order_lines,total_sales,total_profit,average_profit
0,0.00,4798,1087908.47,320987.60,66.90
1,0.10,94,54369.35,9029.18,96.06
2,0.15,52,27558.52,1418.99,27.29
3,0.20,3657,764594.37,90337.31,24.70
4,0.30,227,103226.66,-10369.28,-45.68
5,0.32,27,14493.46,-2391.14,-88.56
6,0.40,206,116417.78,-23057.05,-111.93
7,0.45,11,5484.97,-2493.11,-226.65
8,0.50,66,58918.54,-20506.43,-310.70
9,0.60,138,6644.70,-5944.66,-43.08


In [36]:
cleaning_log=pd.DataFrame([
    {
        'Issue':'Leading/trailing whitespace in Product Name',
        'Action':'Removed leading/trailing whitespace',
        'Reason':'Prevent inconsistent product-name grouping',
        'Rows affected':16
    },
    {
        'Issue':'Order Date stored as object',
        'Action':'Converted to datetime',
        'Reason':'Enable reliable date operations',
        'Rows affected':len(df)
    },
    {
        'Issue':'Ship Date stored as object',
        'Action':'Converted to datetime',
        'Reason':'Enable reliable date operations',
        'Rows affected':len(df)
    },
    {
        'Issue':'Postal Code stored as numeric',
        'Action':'Converted to string',
        'Reason':'Treat postal codes as identifiers, not quantities',
        'Rows affected':len(df)
    }
])

display(cleaning_log)
cleaning_log.to_csv('../reports/cleaning_log.csv',index=False)
print('Cleaning log updated and saved.')

,Issue,Action,Reason,Rows affected
0,Leading/trailing whitespace in Product Name,Removed leading/trailing whitespace,Prevent inconsistent product-name grouping,16
1,Order Date stored as object,Converted to datetime,Enable reliable date operations,9994
2,Ship Date stored as object,Converted to datetime,Enable reliable date operations,9994
3,Postal Code stored as numeric,Converted to string,"Treat postal codes as identifiers, not quantities",9994


Cleaning log updated and saved.


In [37]:
display(pd.read_csv('../reports/validation.csv'))

,Metric,Before cleaning,After cleaning,Difference
0,Row count,9.994000e+03,9.994000e+03,0.0
1,Total sales,2.297201e+06,2.297201e+06,0.0


In [38]:
import sys
from pathlib import Path

sys.path.insert(0,str(Path.cwd().parent))

from src.cleaning import clean_superstore

print('Function imported successfully!')

Function imported successfully!


In [39]:
raw_df=pd.read_csv('../data/raw/Sample - Superstore.csv',encoding='cp1252')
cleaned_df=clean_superstore(raw_df)

print('raw shape:',raw_df.shape)
print('cleaned shape:',cleaned_df.shape)
print()
print(cleaned_df[['Order Date','Ship Date','Postal Code','Product Name']].dtypes)

raw shape: (9994, 21)
cleaned shape: (9994, 21)

Order Date      datetime64[ns]
Ship Date       datetime64[ns]
Postal Code     string[python]
Product Name            object
dtype: object


In [40]:
print(
    'Extra spaces remaining:',
    (
        cleaned_df['Product Name'].notna()
        & (cleaned_df['Product Name'] != cleaned_df['Product Name'].str.strip())
    ).sum()
)
print(
    'Raw dates still strings:',
    raw_df['Order Date'].dtype == 'object'
)
print(
    'Original DataFrame unchanged:',
    raw_df['Order Date'].dtype == 'object'
    and raw_df.shape == cleaned_df.shape
)

Extra spaces remaining: 0
Raw dates still strings: True
Original DataFrame unchanged: True


In [41]:
print('missing values:',cleaned_df.isna().sum().sum())
print('exact duplicate rows:',cleaned_df.duplicated().sum())
print('total rows:',len(cleaned_df))
print('total columns:',len(cleaned_df.columns))

missing values: 0
exact duplicate rows: 0
total rows: 9994
total columns: 21
